In [1]:
import pandas as pd
import time
import requests
import csv
import json
import ast
import re
import numpy as np
import html
import networkx as nx
import polars as pl


from datasketch import MinHash, MinHashLSH, LeanMinHash
from tqdm import tqdm
import kshingle as ks
import pickle
from tqdm import tqdm
import ahocorasick

from gliner import GLiNER
import unicodedata

def remove_accents(text):
    # Normalize the string to decompose characters with accents
    normalized_text = unicodedata.normalize('NFD', text)
    # Filter out accent marks
    return ''.join(char for char in normalized_text if unicodedata.category(char) != 'Mn')


def merge_entities(entities):
    if not entities:
        return []
    merged = []
    current = entities[0]
    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (next_entity['start'] == current['end'] + 1 or next_entity['start'] == current['end']):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity
    # Append the last entity
    merged.append(current)
    return merged

def get_shingles(text, shingle_length, set_range=False, wildcard=False):
    if set_range:
        shingles = ks.shingleset_range(text, shingle_length-1, shingle_length)
    else:
        shingles = ks.shingleset_k(text, shingle_length)
    if wildcard:
        shingles = shingles.union(ks.wildcard_shinglesets(shingles, 2))
    return shingles


# Define a function to create a MinHash object from a string
def create_minhash(text, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for word in get_shingles(text, 3, wildcard=False):
        m.update(word.encode('utf-16', 'surrogatepass').decode('utf-16').encode('utf-8'))
    return LeanMinHash(m)

mailto = "jacd@itu.dk"

def query_openalex_funders(search_string, abbrev = False):
    if abbrev:
        search_string = f'"{search_string}"'
    url = f'https://api.openalex.org/funders?mailto={mailto}&search={search_string}'
    response = requests.get(url)
    time.sleep(0.1)
    if response.status_code == 200:
        data = response.json()
        if 'results' in data and len(data['results']) > 0:
            result = data['results']
            for r in result:
                r['search_string'] = search_string
            return result
        else:
            return [{"search_string": search_string,
                "error": "No results found"}]
    else:
        print(f"Error fetching data for batch. Status code: {response.status_code}")
        return [{"search_string": search_string,
            "error": "Failed to fetch data"}]

def remove_surrogates(s):
    if isinstance(s, str):
        return s.encode('utf-8', 'replace').decode('utf-8')
    return s


tqdm.pandas()

/opt/homebrew/anaconda3/envs/emotions/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

with open("openalex_institutions_new/funders_core+basic+ids+extra+counts.json") as f:
    funder_schema = json.load(f)

dtype_dict = {k: v['type'] for k, v in funder_schema['schema'].items()}
na_values = {k: v['NA value'] for k, v in funder_schema['schema'].items()}
dtype_dict['alternate_titles'] = "str"


filepath = "openalex_institutions_new/funders_core+basic+ids+extra+counts.tsv"

funders = pd.read_csv(filepath, sep="\t", encoding='utf-8', dtype=dtype_dict, na_values=na_values, doublequote=False,
             escapechar="\\",on_bad_lines="warn", low_memory=False, quoting=csv.QUOTE_MINIMAL)

funders['alternate_titles'] = funders['alternate_titles'].apply(ast.literal_eval)

with open('openalex_institutions_new/institutions_core+basic+alternate_names+ids+geo+urls+roles+metrics.json') as f:
    institution_schema = json.load(f)

dtype_dict = {k: v['type'] for k, v in institution_schema['schema'].items()}
na_values = {k: v['NA value'] for k, v in institution_schema['schema'].items()}
dtype_dict['display_name_acronyms'] = "str"
dtype_dict['display_name_alternatives'] = "str"
dtype_dict['roles'] = "str"

filepath = "openalex_institutions_new/institutions_core+basic+alternate_names+ids+geo+urls+roles+metrics.tsv"

institutions = pd.read_csv(filepath, sep="\t", encoding='utf-8', dtype=dtype_dict, na_values=na_values, doublequote=False,
             escapechar="\\",on_bad_lines="warn", low_memory=False, quoting=csv.QUOTE_MINIMAL)

institutions['display_name_alternatives'] = institutions['display_name_alternatives'].apply(ast.literal_eval)
institutions['display_name_acronyms'] = institutions['display_name_acronyms'].apply(ast.literal_eval)


In [4]:
import pandas as pd
import numpy as np
import json
import tldextract
import re

def norm_str(s):
    if pd.isna(s) or s == "":
        return np.nan
    return str(s).strip()

def norm_id_url_to_id(s):
    s = norm_str(s)
    if pd.isna(s): return s
    # Strip common OpenAlex/GRID/ROR/Wikidata URL prefixes to bare IDs where possible
    s = re.sub(r'https?://(www\.)?', '', s.lower())
    s = s.replace('openalex.org/', '').replace('ror.org/', '').replace('www.wikidata.org/wiki/', '')
    return s

def reg_domain(url):
    url = norm_str(url)
    if pd.isna(url): return np.nan
    ext = tldextract.extract(url)
    if not ext.registered_domain:
        return np.nan
    return ext.registered_domain.lower()


# --- Load your two dataframes ---
# df1: the 32,433-row table (looks like funders: has grants_count)
# df2: the 114,713-row table (OpenAlex institutions-like, with roles json)
# df1 = pd.read_parquet('left.parquet'); df2 = pd.read_parquet('right.parquet')

df1 = funders
df2 = institutions

# Normalize key fields
for col in ['ids:ror','ids:wikidata','homepage_url','display_name','country_code']:
    if col in df1.columns: df1[col] = df1[col].map(norm_str)
    if col in df2.columns: df2[col] = df2[col].map(norm_str)

# Keep registered domains
if 'homepage_url' in df1: df1['homepage_domain'] = df1['homepage_url'].map(reg_domain)
if 'homepage_url' in df2: df2['homepage_domain'] = df2['homepage_url'].map(reg_domain)

# Normalize IDs for exact matches
for col in ['ids:ror','ids:wikidata']:
    if col in df1: df1[col+'_norm'] = df1[col].map(norm_id_url_to_id)
    if col in df2: df2[col+'_norm'] = df2[col].map(norm_id_url_to_id)

# Prepare alt/aka sets
def to_set(x):
    if isinstance(x, list):
        return {str(s).casefold() for s in x if str(s).strip() != ''}
    if isinstance(x, str) and x.strip() != '':
        try:
            arr = json.loads(x)
            if isinstance(arr, list):
                return {str(s).casefold() for s in arr if str(s).strip() != ''}
        except Exception:
            pass
    return set()

df1['name_main'] = df1['display_name'].str.casefold()
df1['name_alts'] = df1.get('alternate_titles', pd.Series([[]]*len(df1))).map(to_set)

df2['name_main'] = df2['display_name'].str.casefold()
alts2 = df2.get('display_name_alternatives', pd.Series([[]]*len(df2))).map(to_set)
acrs2 = df2.get('display_name_acronyms', pd.Series([[]]*len(df2))).map(to_set)
df2['name_alts'] = [a.union(b) for a,b in zip(alts2, acrs2)]

# --- Parse roles from df2 ---
def parse_roles(cell):
    if isinstance(cell, list):
        return cell
    if isinstance(cell, str) and cell.strip():
        try:
            return json.loads(cell)
        except Exception:
            return []
    return []

df2['roles_list'] = df2['roles'].map(parse_roles)

def extract_role_flags_and_ids(roles):
    flags = {'has_funder_role': False, 'has_publisher_role': False, 'has_institution_role': False}
    funder_ids, publisher_ids, institution_ids = set(), set(), set()
    for r in roles:
        role = r.get('role')
        rid = r.get('id', '')
        # Keep bare OpenAlex id if present
        bare = norm_id_url_to_id(rid) if rid else None
        if role == 'funder':
            flags['has_funder_role'] = True
            if bare and bare.startswith('f'): funder_ids.add(bare)
        elif role == 'publisher':
            flags['has_publisher_role'] = True
            if bare and bare.startswith('p'): publisher_ids.add(bare)
        elif role == 'institution':
            flags['has_institution_role'] = True
            if bare and bare.startswith('i'): institution_ids.add(bare)
    return pd.Series({
        **flags,
        'openalex_funder_ids': list(funder_ids),
        'openalex_publisher_ids': list(publisher_ids),
        'openalex_institution_ids': list(institution_ids),
    })

/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_85125/2671607478.py:24: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if not ext.registered_domain:
/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_85125/2671607478.py:26: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return ext.registered_domain.lower()
/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_85125/2671607478.py:24: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if not ext.registered_domain:
/var/f

In [5]:
df2 = pd.concat([df2, df2['roles_list'].apply(extract_role_flags_and_ids)], axis=1)


# For funder-targeted join (adjust for your role)
cand_cols = [
    'id', 'ids:ror_norm', 'ids:wikidata_norm',
    'homepage_domain', 'country_code', 'display_name',
    'name_main', 'has_funder_role', 'openalex_funder_ids'
]
candidates = df2.loc[:, [c for c in cand_cols if c in df2.columns]].copy()
candidates = candidates[candidates['has_funder_role'] == True]

left_cols = [
    'id', 'ids:ror_norm', 'ids:wikidata_norm',
    'homepage_domain', 'country_code',
    'display_name', 'name_main'
]
left = df1.loc[:, [c for c in left_cols if c in df1.columns]].copy()


In [6]:
# Build maps (dropna avoids None keys)
map_ror = dict(zip(candidates['ids:ror_norm'].dropna(), candidates['id']))
map_wd  = dict(zip(candidates['ids:wikidata_norm'].dropna(), candidates['id']))

left['match_ror'] = left['ids:ror_norm'].map(map_ror)
left['match_wd']  = left['ids:wikidata_norm'].map(map_wd)

# Prefer ROR, then Wikidata
left['match_t1'] = left['match_ror'].fillna(left['match_wd'])
t1_links = left.loc[left['match_t1'].notna(), ['id','match_t1']].rename(columns={'id':'id_left','match_t1':'id_right'})
resolved_ids = set(t1_links['id_left'])


In [7]:
# Work only on unresolved rows
rem = left[~left['id'].isin(resolved_ids)].copy()

# Index on the join keys (small projection on right)
cand_dom = candidates.loc[:, ['id','homepage_domain','country_code']].dropna(subset=['homepage_domain']).copy()
cand_dom = cand_dom.set_index(['homepage_domain','country_code'])

# Prepare left as index too
rem2 = rem.dropna(subset=['homepage_domain','country_code']).copy()
rem2 = rem2.set_index(['homepage_domain','country_code'])

# Join only the 'id' from the right
j = rem2.join(cand_dom[['id']].rename(columns={'id':'id_right'}), how='left')

t2 = j.reset_index()
t2_links = t2.loc[t2['id_right'].notna(), ['id','id_right']].rename(columns={'id':'id_left'})
resolved_ids |= set(t2_links['id_left'])


In [8]:
# Remaining rows after T1+T2
rem3 = left[~left['id'].isin(resolved_ids)][['id','display_name','name_main','country_code','homepage_domain']].copy()

# Build compact candidate index (no alts here)
cand_min = candidates[['id','display_name','name_main','country_code','homepage_domain']].copy()
cand_min['first_letter'] = cand_min['name_main'].str[:1]
rem3['first_letter'] = rem3['name_main'].str[:1]

# Pre-group right side to iterate small pools
right_groups = dict(tuple(cand_min.groupby(['country_code','first_letter'], dropna=False)))

import re
def jaccard(a, b):
    if not a or not b: return 0.0
    ta, tb = set(re.findall(r'\w+', str(a).lower())), set(re.findall(r'\w+', str(b).lower()))
    if not ta or not tb: return 0.0
    return len(ta & tb) / len(ta | tb)

rows = []
for _, L in rem3.iterrows():
    key = (L.get('country_code'), L.get('first_letter'))
    pool = right_groups.get(key, cand_min)  # fall back to all if missing (rare)
    best_id, best_s = None, -1
    for _, R in pool.iterrows():
        s = 0
        if pd.notna(L['name_main']) and L['name_main'] == R['name_main']:
            s = 100
        else:
            s = int(25 * jaccard(L['display_name'], R['display_name']))
        if L.get('homepage_domain') and R.get('homepage_domain'):
            if L['homepage_domain'] == R['homepage_domain']:
                s += 10
        if s > best_s:
            best_s, best_id = s, R['id']
    if best_s >= 100 and best_id is not None:
        rows.append((L['id'], best_id, best_s))

t3_links = pd.DataFrame(rows, columns=['id_left','id_right','score'])


In [9]:
links = (
    pd.concat([
        t1_links.assign(method='id_exact'),
        t2_links.assign(method='domain_cc'),
        t3_links[['id_left','id_right']].assign(method='name_score')
    ], ignore_index=True)
    .drop_duplicates(subset=['id_left'])
)

# Optional: release intermediates
del rem, rem2, rem3, cand_dom, cand_min, right_groups, t1_links, t2_links, t3_links
import gc; gc.collect()


43

In [11]:
import pandas as pd, json, re

def to_set(x):
    if isinstance(x, list): return {str(s).strip() for s in x if str(s).strip()}
    if isinstance(x, str) and x.strip():
        try:
            arr = json.loads(x)
            if isinstance(arr, list):
                return {str(s).strip() for s in arr if str(s).strip()}
        except Exception:
            return {x.strip()}
    return set()

def merge_sets(*sets):
    out=set()
    for s in sets:
        if isinstance(s, (set, list, tuple)):
            out.update(str(v).strip() for v in s if str(v).strip())
    return out

def as_json_or_empty(s):  # deterministic order
    return "" if not s else json.dumps(sorted(set(s)))

# Optional: normalize acronyms (upper, drop long tokens, keep A–Z&0–9 only)
def normalize_acronyms(acr_set, max_len=15):
    out=set()
    for a in acr_set or []:
        a = re.sub(r'[^0-9A-Za-z&.-]', '', str(a)).upper()
        if a and len(a) <= max_len:
            out.add(a)
    return out

names_cols = ['id','display_name','display_name_alternatives','display_name_acronyms',
              'country_code','geo:country_code','homepage_url','ids:ror','ids:wikidata']
df2_names = df2.loc[:, [c for c in names_cols if c in df2.columns]].copy()

alts2 = df2_names.get('display_name_alternatives', pd.Series([[]]*len(df2_names))).map(to_set)
acrs2 = df2_names.get('display_name_acronyms',    pd.Series([[]]*len(df2_names))).map(lambda s: normalize_acronyms(to_set(s)))

# union for alternate_titles should include: primary name + alts + acronyms
df2_names['alts_union_set'] = [merge_sets({n}, a, ac) for n,a,ac in
                               zip(df2_names['display_name'], alts2, acrs2)]
df2_names['acronyms_set']   = acrs2

# Thin projection from df1
funder_cols = ['id','display_name','alternate_titles','country_code','homepage_url',
               'ids:ror','ids:wikidata','ids:crossref','ids:doi',
               'grants_count','works_count','cited_by_count','roles','description']
funders_min = df1.loc[:, [c for c in funder_cols if c in df1.columns]].copy()

funder_alts = funders_min.get('alternate_titles', pd.Series([[]]*len(funders_min))).map(to_set)

# Join to df2 enrichment sets
enrich = (
    links[['id_left','id_right']]
    .merge(df2_names[['id','alts_union_set','acronyms_set']], left_on='id_right', right_on='id', how='left', sort=False)
    .drop(columns=['id'])
)

# Aggregate per funder (in case of multiple matches)
agg = enrich.groupby('id_left', as_index=False).agg({
    'alts_union_set': lambda lst: merge_sets(*lst),
    'acronyms_set':   lambda lst: merge_sets(*lst),
})

# Attach to funders and build both columns
funders_min = funders_min.merge(funder_alts.rename('funder_alts_set'), left_index=True, right_index=True, how='left')
funders_min = funders_min.merge(agg, left_on='id', right_on='id_left', how='left', sort=False).drop(columns=['id_left'])

# alternate_titles = union of (primary name) + existing alternates + df2 alts + df2 acronyms
funders_min['alternate_titles'] = [
    as_json_or_empty(merge_sets({name}, fset or set(), a2 or set(), ac2 or set()))
    for name, fset, a2, ac2 in zip(
        funders_min['display_name'], funders_min['funder_alts_set'],
        funders_min['alts_union_set'], funders_min['acronyms_set'])
]

# separate acronyms column as JSON array (normalized, deduped)
funders_min['acronyms'] = [
    as_json_or_empty(normalize_acronyms(merge_sets(ac2)))
    for ac2 in funders_min['acronyms_set']
]

funders_enriched = funders_min.drop(columns=['funder_alts_set','alts_union_set','acronyms_set'])

used_right_ids = set(links['id_right'])
unmatched = df2[~df2['id'].isin(used_right_ids)].copy()

alts_unm = unmatched.get('display_name_alternatives', pd.Series([[]]*len(unmatched))).map(to_set)
acrs_unm = unmatched.get('display_name_acronyms',    pd.Series([[]]*len(unmatched))).map(lambda s: normalize_acronyms(to_set(s)))

alternate_titles_json = [
    as_json_or_empty(merge_sets({n}, a, ac))
    for n,a,ac in zip(unmatched['display_name'], alts_unm, acrs_unm)
]
acronyms_json = [as_json_or_empty(ac) for ac in acrs_unm]

unmatched_out = pd.DataFrame({
    'id':               unmatched['id'],  # if id-space conflicts, set -1 and keep df2 id in 'source_institution_id'
    'display_name':     unmatched['display_name'],
    'country_code':     unmatched.get('country_code', unmatched.get('geo:country_code', "")),
    'roles':            unmatched.get('roles', ""),
    'ids:wikidata':     unmatched.get('ids:wikidata', ""),
    'ids:ror':          unmatched.get('ids:ror', ""),
    'ids:crossref':     "",
    'ids:doi':          "",
    'alternate_titles': alternate_titles_json,
    'acronyms':         acronyms_json,
    'description':      "",
    'homepage_url':     unmatched.get('homepage_url', ""),
    'grants_count':     -1,
    'works_count':      unmatched.get('summary_stats:works_count', unmatched.get('works_count', -1)).fillna(-1),
    'cited_by_count':   unmatched.get('summary_stats:cited_by_count', -1).fillna(-1),
    'source':           'institutions',
})
funders_enriched['source'] = 'funders'

final_cols = ['id','display_name','country_code','roles',
              'ids:wikidata','ids:ror','ids:crossref','ids:doi',
              'alternate_titles','acronyms','description','homepage_url',
              'grants_count','works_count','cited_by_count','source']

# Ensure columns exist & order
for c in final_cols:
    if c not in funders_enriched.columns:
        funders_enriched[c] = "" if c not in ['grants_count','works_count','cited_by_count'] else -1
    if c not in unmatched_out.columns:
        unmatched_out[c]    = "" if c not in ['grants_count','works_count','cited_by_count'] else -1

final_funders = pd.concat([funders_enriched[final_cols].sort_values(by='grants_count', ascending=False),
                            unmatched_out[final_cols]], ignore_index=True)


In [15]:
ror = pd.read_csv('v1.72-2025-10-06-ror-data_schema_v2.csv')
#in_dimensions = pl.read_parquet('funders_dimensions.parquet').with_columns(pl.lit('True').alias('in_dimensions')).to_pandas()

#ror = ror.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')

institutions_ = final_funders.merge(ror[['id','external_ids.type.grid.preferred',
                                         'locations.geonames_details.country_code']],
                                           left_on="ids:ror", right_on='id', how='left', suffixes=("","_ror"))

def reg_domain(url):
    if pd.isna(url) or url=="":
        return ""
    ext = tldextract.extract(str(url))
    return (ext.registered_domain or "").lower()

def set_to_json(s):
    return "" if not s else json.dumps(sorted(s))

def coalesce(*vals, empty=""):
    for v in vals:
        if pd.notna(v) and v not in ("", []):
            return v
    return empty

inst = institutions_.copy()

# coalesce country (your snippet did this inline; keep it as a column)
inst["country_code_final"] = inst["country_code"].fillna(inst["locations.geonames_details.country_code"])
inst["homepage_domain"] = inst["homepage_url"].map(reg_domain)

# normalized key pieces
inst["name_norm"] = inst["display_name"].str.casefold().str.strip()

# prefer (name, country, domain) when domain exists; fallback to (name, country)
inst["group_key"] = np.where(
    inst["homepage_domain"].ne(""),
    inst["name_norm"] + " | " + inst["country_code_final"].fillna("") + " | " + inst["homepage_domain"],
    inst["name_norm"] + " | " + inst["country_code_final"].fillna("")
)

# booleans/weights for ranking
inst["is_funder"] = (inst.get("source","") == "funders").astype(int)
inst["has_ror"] = inst["ids:ror"].fillna("").astype(bool).astype(int)
inst["has_wd"]  = inst["ids:wikidata"].fillna("").astype(bool).astype(int)
inst["has_domain"] = inst["homepage_domain"].fillna("").ne("").astype(int)

# replace -1 with NaN before ranking numeric richness
for c in ["grants_count","works_count","cited_by_count"]:
    if c in inst.columns:
        inst[c+"_nz"] = inst[c].replace({-1: np.nan})
    else:
        inst[c+"_nz"] = np.nan

# sort order (higher is better)
sort_cols = [
    "is_funder","has_ror","has_wd","has_domain",
    "grants_count_nz","works_count_nz","cited_by_count_nz"
]
inst = inst.sort_values(by=sort_cols, ascending=False, kind="stable")


keep_cols_fixed = [
    "id","display_name","country_code_final","roles","ids:wikidata","ids:ror",
    "ids:crossref","ids:doi","description","homepage_url"
]

def reduce_group(g: pd.DataFrame) -> pd.Series:
    # canonical is the first after sorting above
    canon = g.iloc[0]

    # union alternates and acronyms (also include all observed display_names)
    alt_sets = [to_set(x) for x in g.get("alternate_titles","")]
    alt_sets.append(set(g["display_name"].dropna().astype(str).str.strip()))
    acr_sets = [normalize_acronyms(to_set(x)) for x in g.get("acronyms","")]
    acr_union = set().union(*acr_sets)
    alt_union = set().union(*alt_sets, acr_union)  # acronyms inside alternates too

    # other IDs (if you want to preserve)
    other_ror = set(g["ids:ror"].dropna().astype(str)) - ({canon.get("ids:ror")} if pd.notna(canon.get("ids:ror")) else set())
    other_wd  = set(g["ids:wikidata"].dropna().astype(str)) - ({canon.get("ids:wikidata")} if pd.notna(canon.get("ids:wikidata")) else set())

    # numeric fields: take max (non-missing)
    def max_or_neg1(series):
        vals = pd.to_numeric(series, errors="coerce").dropna()
        return int(vals.max()) if len(vals) else -1

    out = {
        # fixed (canonical)
        "id": canon.get("id"),
        "display_name": canon.get("display_name"),
        "country_code": canon.get("country_code_final") or "",
        "roles": canon.get("roles",""),
        "ids:wikidata": coalesce(canon.get("ids:wikidata"), ""),
        "ids:ror":      coalesce(canon.get("ids:ror"), ""),
        "ids:crossref": coalesce(canon.get("ids:crossref"), ""),
        "ids:doi":      coalesce(canon.get("ids:doi"), ""),
        "description":  coalesce(canon.get("description"), ""),
        "homepage_url": coalesce(canon.get("homepage_url"), ""),

        # unions
        "alternate_titles": set_to_json(alt_union),
        "acronyms":         set_to_json(acr_union),

        # optional provenance
        "sources_merged":   json.dumps(sorted(set(g.get("source","institutions").astype(str)))),

        # numeric merged
        "grants_count":     max_or_neg1(g.get("grants_count", pd.Series(dtype=float))),
        "works_count":      max_or_neg1(g.get("works_count", pd.Series(dtype=float))),
        "cited_by_count":   max_or_neg1(g.get("cited_by_count", pd.Series(dtype=float))),

        # optional: keep “other ids” for audit
        "other_ids:ror":    set_to_json(other_ror),
        "other_ids:wikidata": set_to_json(other_wd),
        "homepage_domain":  canon.get("homepage_domain",""),
    }
    return pd.Series(out)

collapsed = inst.groupby("group_key", sort=False, group_keys=False).apply(reduce_group).reset_index(drop=True)


/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_85125/1367014506.py:1: DtypeWarning: Columns (5,14) have mixed types. Specify dtype option on import or set low_memory=False.
  ror = pd.read_csv('v1.72-2025-10-06-ror-data_schema_v2.csv')
/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_85125/1367014506.py:14: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  return (ext.registered_domain or "").lower()


In [27]:
inst_test = pd.read_parquet('inst.parquet')

In [17]:
inst_test

,id,display_name,country_code,roles,ids:wikidata,ids:ror,ids:crossref,ids:doi,alternate_titles,acronyms,...,homepage_domain,name_norm,group_key,is_funder,has_ror,has_wd,has_domain,grants_count_nz,works_count_nz,cited_by_count_nz
0,4320321001,National Natural Science Foundation of China,CN,"[{'role': 'institution', 'id': 'https://openal...",https://www.wikidata.org/entity/Q4501917,https://ror.org/01h0zpd94,501100001809,https://doi.org/10.13039/501100001809,"[""Chinese National Science Foundation"", ""Gu\u0...","[""NSFC""]",...,nsfc.gov.cn,national natural science foundation of china,national natural science foundation of china |...,1,1,1,1,717144.0,2131044,42764999
1,4320332161,National Institutes of Health,US,"[{'role': 'funder', 'id': 'https://openalex.or...",https://www.wikidata.org/entity/Q390551,https://ror.org/01cwqze88,100000002,https://doi.org/10.13039/100000002,"[""Institutos Nacionales de la Salud"", ""NIH"", ""...","[""NIH""]",...,nih.gov,national institutes of health,national institutes of health | US | nih.gov,1,1,1,1,271804.0,383535,15025659
2,4320306076,National Science Foundation,US,"[{'role': 'publisher', 'id': 'https://openalex...",https://www.wikidata.org/entity/Q304878,https://ror.org/021nxhr62,100000001,https://doi.org/10.13039/100000001,"[""NSF"", ""National Science Foundation"", ""U.S. N...","[""NSF""]",...,nsf.gov,national science foundation,national science foundation | US | nsf.gov,1,1,1,1,207831.0,391743,13017807
3,4320334764,Japan Society for the Promotion of Science,JP,"[{'role': 'funder', 'id': 'https://openalex.or...",https://www.wikidata.org/entity/Q199781,https://ror.org/00hhkn466,501100001691,https://doi.org/10.13039/501100001691,"[""Grants-in-Aid for Scientific Research"", ""Gra...","[""JSPS""]",...,jsps.go.jp,japan society for the promotion of science,japan society for the promotion of science | J...,1,1,1,1,157091.0,216361,3978797
4,4320320879,Deutsche Forschungsgemeinschaft,DE,"[{'role': 'institution', 'id': 'https://openal...",https://www.wikidata.org/entity/Q707283,https://ror.org/018mejw64,501100001659,https://doi.org/10.13039/501100001659,"[""DFG"", ""Deutsche Forschungsgemeinschaft"", ""Ge...","[""DFG""]",...,dfg.de,deutsche forschungsgemeinschaft,deutsche forschungsgemeinschaft | DE | dfg.de,1,1,1,1,123340.0,240248,6621838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131935,4389425357,Fondation Voir & Entendre,None,"[{""role"":""institution"",""id"":""https:\/\/openale...",None,https://ror.org/01ph0t361,,,"[""Fondation Voir & Entendre"", ""Voir & Entendre...",,...,None,fondation voir & entendre,fondation voir & entendre | FR,0,1,0,0,NaN,0,0
131936,4391768288,State Key Laboratory of Pest Control and Resou...,None,"[{""role"":""institution"",""id"":""https:\/\/openale...",None,https://ror.org/01bs5br94,,,"[""State Key Lab of Pest Control and Resource U...",,...,None,state key laboratory of pest control and resou...,state key laboratory of pest control and resou...,0,1,0,0,NaN,0,0
131952,4402554148,Instituto Educativo Dolores Del Rio Extensión ...,None,"[{""role"":""institution"",""id"":""https:\/\/openale...",None,https://ror.org/00q1dg558,,,"[""IEDR Santiago Papasquiaro"", ""IEDRAC"", ""Insti...","[""IEDRAC""]",...,None,instituto educativo dolores del rio extensión ...,instituto educativo dolores del rio extensión ...,0,1,0,0,NaN,0,0
131955,4402554192,Bureau of Public Roads,None,"[{""role"":""institution"",""id"":""https:\/\/openale...",None,https://ror.org/039c93x30,,,"[""Bureau of Public Roads"", ""Office of Public R...",,...,None,bureau of public roads,bureau of public roads | US,0,1,0,0,NaN,0,0


In [34]:
(collapsed['all_names'] == collapsed_test['all_names']).all()

True

In [23]:
collapsed_test = pd.read_parquet('collapsed.parquet')

In [26]:
inst_test = pd.read_parquet('artifacts/institutions_normalized.parquet')

In [46]:
high_aug = pd.read_parquet('artifacts/u_agencies_high_augmented.parquet')

In [30]:
def add_normalized_name_lists(collapsed: pd.DataFrame) -> pd.DataFrame:
    out = collapsed.copy()
    out["all_names"] = out.apply(lambda x: [x["display_name"]] + ast.literal_eval(x["alternate_titles"]), axis=1)
    out["all_names"] = out["all_names"].apply(lambda xs: [y.replace("&", "and") for y in xs])
    out["all_names"] = out["all_names"].apply(lambda xs: [y.encode("utf-16", "surrogatepass").decode("utf-16") for y in xs])
    out["all_names"] = out["all_names"].apply(lambda xs: [remove_accents(y) for y in xs])
    out["all_names"] = out["all_names"].apply(lambda xs: [y.replace("’", "'") for y in xs])
    out["all_names_lower"] = out["all_names"].apply(lambda xs: [y.lower() for y in xs])
    out["all_names_lower"] = out["all_names_lower"].apply(lambda xs: [y.encode("utf-16", "surrogatepass").decode("utf-16") for y in xs])
    out["all_names_lower"] = out["all_names_lower"].apply(lambda xs: [y.strip() for y in xs])
    out["display_name"] = out["display_name"].apply(remove_accents)
    out["display_name_lower"] = out["display_name"].str.lower()
    out["display_name_lower"] = out["display_name_lower"].apply(lambda x: x.encode("utf-16", "surrogatepass").decode("utf-16"))
    out["display_name_lower"] = out["display_name_lower"].str.strip()
    out["acronyms"] = out["acronyms"].apply(lambda x: x if x else "[]")
    out["acronyms"] = out["acronyms"].apply(ast.literal_eval)
    out["acronyms"] = out["acronyms"].apply(lambda xs: [y.encode("utf-16", "surrogatepass").decode("utf-16") for y in xs])
    out["acronyms"] = out["acronyms"].apply(lambda xs: [y.strip() for y in xs])
    out["all_names"] = out["all_names"].apply(lambda xs: list(set(xs)))
    out["all_names_lower"] = out["all_names_lower"].apply(lambda xs: list(set(xs)))
    out["acronyms"] = out["acronyms"].apply(lambda xs: list(set(xs)))
    return out

collapsed_test = add_normalized_name_lists(collapsed_test)

In [31]:
collapsed['all_names'] = collapsed.apply(lambda x: [x['display_name']] 
                                                 + ast.literal_eval(x['alternate_titles']), axis=1)
collapsed['all_names'] = collapsed['all_names'].apply(lambda x: [y.replace('&', 'and') for y in x])
collapsed['all_names'] = collapsed['all_names'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])

collapsed['all_names'] = collapsed['all_names'].apply(
    lambda x: [remove_accents(y) for y in x])

collapsed['all_names'] = collapsed['all_names'].apply(
    lambda x: [y.replace("’", "'") for y in x])


collapsed['all_names_lower'] = collapsed['all_names'].apply(lambda x: [y.lower() for y in x])
collapsed['all_names_lower'] = collapsed['all_names_lower'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])
collapsed['all_names_lower'] = collapsed['all_names_lower'].apply(
    lambda x: [y.strip() for y in x])

collapsed['display_name'] = collapsed['display_name'].apply(remove_accents)
collapsed['display_name_lower'] = collapsed['display_name'].str.lower()
collapsed['display_name_lower'] = collapsed['display_name_lower'].apply(
    lambda x: x.encode('utf-16', 'surrogatepass').decode('utf-16'))
collapsed['display_name_lower'] = collapsed['display_name_lower'].str.strip()

collapsed['acronyms'] = collapsed['acronyms'].apply(lambda x: x if x else "[]")
collapsed['acronyms'] = collapsed['acronyms'].apply(ast.literal_eval)
collapsed['acronyms'] = collapsed['acronyms'].apply(
    lambda x: [y.encode('utf-16', 'surrogatepass').decode('utf-16') for y in x])
collapsed['acronyms'] = collapsed['acronyms'].apply(
    lambda x: [y.strip() for y in x])


collapsed['all_names'] = collapsed['all_names'].apply(lambda x: list(set(x)))
collapsed['all_names_lower'] = collapsed['all_names_lower'].apply(lambda x: list(set(x)))
collapsed['acronyms'] = collapsed['acronyms'].apply(lambda x: list(set(x)))



In [36]:
collapsed

,id,display_name,country_code,roles,ids:wikidata,ids:ror,ids:crossref,ids:doi,description,homepage_url,...,sources_merged,grants_count,works_count,cited_by_count,other_ids:ror,other_ids:wikidata,homepage_domain,all_names,all_names_lower,display_name_lower
0,4320321001,National Natural Science Foundation of China,CN,"[{'role': 'institution', 'id': 'https://openal...",https://www.wikidata.org/entity/Q4501917,https://ror.org/01h0zpd94,501100001809,https://doi.org/10.13039/501100001809,Chinese government body,https://www.nsfc.gov.cn,...,"[""funders""]",717144,2131044,42764999,,,nsfc.gov.cn,"[NNSF of China, NSFC, National Nature Science ...","[natural science foundation of china, guojia z...",national natural science foundation of china
1,4320332161,National Institutes of Health,US,"[{'role': 'funder', 'id': 'https://openalex.or...",https://www.wikidata.org/entity/Q390551,https://ror.org/01cwqze88,100000002,https://doi.org/10.13039/100000002,US government medical research agency,http://www.nih.gov,...,"[""funders""]",271804,383535,15025659,,,nih.gov,"[National Institutes of Health, NIH, Instituto...","[national institutes of health, institutos nac...",national institutes of health
2,4320306076,National Science Foundation,US,"[{'role': 'publisher', 'id': 'https://openalex...",https://www.wikidata.org/entity/Q304878,https://ror.org/021nxhr62,100000001,https://doi.org/10.13039/100000001,United States government agency,https://www.nsf.gov,...,"[""funders""]",207831,391743,13017807,,,nsf.gov,"[USA NSF, National Science Foundation, USNSF, ...","[us nsf, u.s. national science foundation, nat...",national science foundation
3,4320334764,Japan Society for the Promotion of Science,JP,"[{'role': 'funder', 'id': 'https://openalex.or...",https://www.wikidata.org/entity/Q199781,https://ror.org/00hhkn466,501100001691,https://doi.org/10.13039/501100001691,administrative agency in Japan,http://www.jsps.go.jp,...,"[""funders""]",157091,216361,3978797,,,jsps.go.jp,"[Grants-in-Aid for Scientific Research, Grants...","[nihon gakujutsu shinko kai, 日本学術振興会, grants-i...",japan society for the promotion of science
4,4320320879,Deutsche Forschungsgemeinschaft,DE,"[{'role': 'institution', 'id': 'https://openal...",https://www.wikidata.org/entity/Q707283,https://ror.org/018mejw64,501100001659,https://doi.org/10.13039/501100001659,"German research funding organisation, German R...",http://www.dfg.de/en,...,"[""funders""]",123340,240248,6621838,,,dfg.de,"[Deutsche Forschungsgemeinschaft, DFG, German ...","[german research foundation, deutsche forschun...",deutsche forschungsgemeinschaft
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131236,4389425357,Fondation Voir & Entendre,FR,"[{""role"":""institution"",""id"":""https:\/\/openale...",,https://ror.org/01ph0t361,,,,,...,"[""institutions""]",-1,0,0,,,,"[Voir et Entendre Foundation, Fondation Voir a...","[fondation voir and entendre, voir et entendre...",fondation voir & entendre
131237,4391768288,State Key Laboratory of Pest Control and Resou...,CN,"[{""role"":""institution"",""id"":""https:\/\/openale...",,https://ror.org/01bs5br94,,,,,...,"[""institutions""]",-1,0,0,,,,[State Key Lab of Pest Control and Resource Ut...,[state key lab of pest control and resource ut...,state key laboratory of pest control and resou...
131238,4402554148,Instituto Educativo Dolores Del Rio Extension ...,MX,"[{""role"":""institution"",""id"":""https:\/\/openale...",,https://ror.org/00q1dg558,,,,,...,"[""institutions""]",-1,0,0,,,,"[Instituto Educativo Dolores del Rio, A. C., I...","[iedrac, instituto educativo dolores del rio, ...",instituto educativo dolores del rio extension ...
131239,4402554192,Bureau of Public Roads,US,"[{""role"":""institution"",""id"":""https:\/\/openale...",,https://ror.org/039c93x30,,,,,...,"[""institutions""]",-1,0,0,,,,"[Public Roads Administration, Bureau of Public...","[bureau of public roads, office of public road...",bureau of public roads


In [ ]:

u_agencies = pl.read_ipc('WOS_TSV_FUNDING_2023.feather')
u_agencies = (u_agencies
.with_columns(agency_name = pl.col("agency_name").str.split(";"))
.explode("agency_name")
.with_columns(agency_name = pl.col("agency_name").str.split(":").list.first().str.strip_chars())
.filter(pl.col('agency_name') != "")
)['agency_name'].value_counts(sort=True).rename({'agency_name':'grant_agency', 'count':'counts'}).to_pandas()

u_agencies['grant_agency'] = u_agencies['grant_agency'].apply(html.unescape)
u_agencies['grant_agency'] = u_agencies['grant_agency'].str.replace('&', 'and')
u_agencies['grant_agency'] = u_agencies['grant_agency'].str.replace("''", "'")

u_agencies['grant_agency_lower'] = u_agencies['grant_agency'].str.lower()
u_agencies['grant_agency_lower'] = u_agencies['grant_agency_lower'].str.strip()

u_agencies[['abbrev']] = u_agencies.grant_agency.str.extract(r'\(([A-Z]*)\)')
u_agencies.shape

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


(7926557, 4)

### Preparing data for document clustering (no need to run)

In [4]:
minhash_file = 'grant_agency_minhashes.pkl'

with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(u_agencies.index, u_agencies.grant_agency), total=len(u_agencies), desc="Querying grant agency"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)

Querying grant agency: 100%|██████████| 7926557/7926557 [2:02:10<00:00, 1081.30it/s]  


In [5]:
minhash_file = 'grant_agency_minhashes.pkl'
# Create the LSH object with threshold for near-duplicates
lsh = MinHashLSH(threshold=0.95, num_perm=128)

with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            lsh.insert(f"{idx}", lean_minhash)
    except EOFError:
        pass 



In [6]:
edge_list_file = 'edge_list.csv'

with open(minhash_file, 'rb') as f_in, open(edge_list_file, 'w', newline='') as f_out:
    writer = csv.writer(f_out)
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            result = lsh.query(lean_minhash)
            if result:
                for res in result:
                    writer.writerow([idx, res])
    except EOFError:
        pass

### Read outputs from document clustering

edge_list is an edge list of the graph induced by calculating document similarity between all documents (i.e. all funder names). Two documents are linked if jaccard similarity is above 0.95.

In [13]:
edge_list_file = 'edge_list.csv'

#read edgelist from file
G = nx.read_edgelist(edge_list_file, delimiter=',', nodetype=int)

G.remove_edges_from(nx.selfloop_edges(G))
#connected components
components = list(nx.connected_components(G))
len(components)

7297223

In [14]:
#map components to ids in the original dataframe
component_map = {}

for idx, component in enumerate(components):
    for node in component:
        component_map[node] = idx

u_agencies['component'] = u_agencies.index.map(component_map)

### Preprocess openalex data

This block preproccesses both OA funder and institution data and combines them
* combine names (canonical names, alternate names)
* replace & with and
* fix encoding issues
    * some characters (emojis) need to be encoded in utf-16 otherwise the deduplication algorithm doesn't work
* remove accents - WOS data is ASCII which has no accents
* openalex has inconsistent use of apostrophes
* if funders and institutions share canonical name they also share acronyms
* if duplicates are present we keep the funder data and supplement with institution data


In [16]:
institutions = collapsed

In [17]:
ror = pd.read_csv('v1.72-2025-10-06-ror-data_schema_v2.csv')
in_dimensions = pl.read_parquet('funders_dimensions.parquet').with_columns(pl.lit('True').alias('in_dimensions')).to_pandas()

ror = ror.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')

institutions_ = institutions.merge(ror[['id','external_ids.type.grid.preferred']], left_on="ids:ror", right_on='id', how='left', suffixes=("","_ror"))
institutions_ = institutions_.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')
institutions_ = institutions_.rename(columns={'funder':'dimensions_grid', })

institutions_['source'] = institutions_.sources_merged.apply(lambda x: ast.literal_eval(x)[0])

/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_80004/2581196642.py:1: DtypeWarning: Columns (5,14) have mixed types. Specify dtype option on import or set low_memory=False.
  ror = pd.read_csv('v1.72-2025-10-06-ror-data_schema_v2.csv')


In [18]:
institutions_[['id', 'display_name', 'country_code', 'ids:ror', 'homepage_url',
'source', 'dimensions_grid']].to_parquet('oa_institutions.parquet')


In [19]:
del df1, df2, unmatched, institutions_, ror, inst, collapsed, final_funders, funders, funders_enriched
del funders_min, alts2, acrs2, alts_unm, acrs_unm

gc.collect()

1512

### Exact match

In [22]:
u_agencies = u_agencies.merge(institutions[['display_name_lower','display_name', 'id']],
    left_on='grant_agency_lower', right_on='display_name_lower', how='left')

# if id not na then "display_exact" use fillna approach
u_agencies.loc[~u_agencies['id'].isna(), 'match_type'] = 'display_exact'

u_agencies = u_agencies.merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
    left_on='grant_agency_lower', right_on='all_names_lower', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'all_exact'


In [23]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(7425486, 10)
0.9848995742500894


0.5969819911671436

### Apply manual annotations by Jacob

In [25]:
#load hhs map from json
with open('hhs_map.json') as f:
    HHS_map = json.load(f)
HHS_map = {k.strip():v for k, v in HHS_map.items()}

p_1000 = pd.read_csv("plus_1000_labelled.csv", sep=";")
p_1000 = p_1000.assign(id = p_1000.id.str.slice(1),
                       grant_agency = p_1000.grant_agency.apply(
                        lambda x: x[1:-1] if x.startswith('"') and x.endswith('"') else x)
).dropna().set_index('grant_agency')['id'].to_dict()


new_p_1000 = pd.read_csv("new_1000_labelled.csv", sep=";").dropna(subset='id')
new_p_1000 = (new_p_1000.assign(id = new_p_1000.id.astype(int))
).set_index('grant_agency')['id'].to_dict()



In [26]:
last_dict = {'Korean Government (MOEHRD)':'F4320327953',
'Scientific and Technical Research Council of Turkey (TUBITAK)':'F4320322626',
'Strategic Priority Research Program (B) of the Chinese Academy of Sciences' : 'F4320321133',
'National High Technology Research and Development Program ("863" Program) of China':'F4320335773',
'Soft and Hybrid Nanotechnology Experimental (SHyNE) Resource':'F4320332170'}

last_dict['MPG, Germany'] = 'https://openalex.org/F4320322434'
last_dict['GlaxoSmithKline Biologicals SA'] = 'https://openalex.org/F4320307773'
last_dict['Merck Research Laboratories'] = 'https://openalex.org/F4320307776'
last_dict['Spanish Participation Group'] = 'https://openalex.org/F4320322930'
last_dict['CNCSIS'] = 'https://openalex.org/F4320322505'
last_dict['Siemens Healthcare'] = 'https://openalex.org/F4320327217'
last_dict['Ruth L. Kirschstein National Research Service Award'] = 'https://openalex.org/F4320332161'
last_dict['MIUR-PRIN'] = 'https://openalex.org/F4320321873'
last_dict['State Key Research Development Program of China'] = 'https://openalex.org/F4320335777'
last_dict['COLCIENCIAS, Colombia'] = 'https://openalex.org/F4320309955'
last_dict['Korean Scientist Group'] = 'https://openalex.org/F4320322349'
last_dict['ARO MURI'] = 'https://openalex.org/F4320333591'
last_dict['University Grant Commission'] = 'https://openalex.org/F4320320767'
last_dict['Centocor'] = 'https://openalex.org/F4320337601'
last_dict['German Participation Group'] = 'https://openalex.org/F4320331606'
last_dict['Interuniversity Attraction Poles Programme'] = 'https://openalex.org/F4320321454'
last_dict['863 Project'] = 'https://openalex.org/F4320335773'
last_dict['NUPTSF'] = 'https://openalex.org/F4320321001'
last_dict['Austrian Science Foundation'] = 'https://openalex.org/F4320321181'
last_dict['Anhui Initiative in Quantum Information Technologies'] = 'https://openalex.org/F4320334897'
last_dict['Vanier Canada Graduate Scholarship'] = 'https://openalex.org/F4320319880'
last_dict['Brazilian Participation Group'] = 'https://openalex.org/F4320321992'
last_dict['French Participation Group'] = 'https://openalex.org/F4320320883'
last_dict['Japan Participation Group'] = 'https://openalex.org/F4320323772'
#last_dict['Neurotrack Technologies'] = 
last_dict['National Natural Science Fundation of China'] = 'https://openalex.org/F4320321001'
last_dict['Nebraska Research Initiative'] = 'https://openalex.org/F4320310264'
last_dict['DIVISION OF CANCER TREATMENT'] = 'https://openalex.org/F4320309684'
#last_dict['EuroImmun'] = 'https://openalex.org/F4320335322'
last_dict['National Key RD Plan'] = 'https://openalex.org/F4320335777'
last_dict['University Grant Commission, New Delhi'] = 'https://openalex.org/F4320320767'
last_dict['NIH-NIGMS'] = 'https://openalex.org/F4320337354'
last_dict['Sloan Digital Sky Survey (SDSS)'] = 'https://openalex.org/F4320306151'
last_dict['Commission of the European Communities'] = 'https://openalex.org/F4320320300'
last_dict['Icelandic Research Fund'] = 'https://openalex.org/F4320321020'
last_dict['Hong Kong Scholars Program'] = 'https://openalex.org/F4320323540'
last_dict['National Magnetic Confinement Fusion Science Program of China'] = 'https://openalex.org/F4320327893'
last_dict['Taishan Scholars Program of Shandong Province'] = 'https://openalex.org/F4320326189'
last_dict['Fundamental Research Grant Scheme'] = 'https://openalex.org/F4320321709'
last_dict['MICINN, Spain'] = 'https://openalex.org/F4320322930'
last_dict['NASA Office of Space Science'] = 'https://openalex.org/F4320306101'
last_dict['Parkinson"s UK'] = 'https://openalex.org/F4320320021'
last_dict['Fundamental Research Grant Scheme (FRGS)'] = 'https://openalex.org/F4320321709'
last_dict['Elan Pharmaceuticals, Inc.'] = 'https://openalex.org/F4320321667'
last_dict['CONICYT, Chile'] = 'https://openalex.org/F4320334812'
last_dict['Australian Ctr Intl Agr Res'] = 'https://openalex.org/F4320334709'
last_dict['JST-CREST'] = 'https://openalex.org/F4320338075'
last_dict['State of Hungary'] = 'https://openalex.org/F4320328860'
last_dict['Alfred P. Sloan Research Fellowship'] = 'https://openalex.org/F4320306151'
last_dict['Russian Fund for Basic Research'] = 'https://openalex.org/F4320321079'
last_dict['Cotton RandD Corp'] = 'https://openalex.org/F4320320417'
last_dict['Merck-Serono'] = 'https://openalex.org/F4320307776'
last_dict['JST CREST, Japan'] = 'https://openalex.org/F4320338075'
last_dict['Brain Korea 21 Project'] = 'https://openalex.org/F4320327953'
last_dict['PAPIIT-UNAM'] = 'https://openalex.org/F4320335100'
last_dict['Taishan Scholar Program of Shandong Province'] = 'https://openalex.org/F4320326189'
last_dict['US Public Health Service'] = 'https://openalex.org/F4320332505'
last_dict['Piramal Imaging'] = 'https://openalex.org/F4320307771'
last_dict['Stanford Graduate Fellowship'] = 'https://openalex.org/F4320308507'
last_dict['Astra-Zeneca'] = 'https://openalex.org/F4320307770'
last_dict['FAPERJ (Brazil)'] = 'https://openalex.org/F4320322749'
last_dict['CAPES, Brazil'] = 'https://openalex.org/F4320321091'
last_dict['European Communities'] = 'https://openalex.org/F4320338352'
last_dict['National Scientific Foundation of China'] = 'https://openalex.org/F4320321001'
last_dict['Sloan Research Fellowship'] = 'https://openalex.org/F4320306151'
last_dict['Fondo de Investigaciones Sanitarias'] = 'https://openalex.org/F4320334923'
last_dict['DIVISION OF HEART AND VASCULAR DISEASES'] = 'https://openalex.org/F4320337338'
last_dict['Division Of Behavioral and Cognitive Sci'] = 'https://openalex.org/F4320337405'
last_dict['COST action'] = 'https://openalex.org/F4320320366'
last_dict['National Science and Technology Pillar Program'] = 'https://openalex.org/F4320335774'
last_dict['National Key Technology Support Program'] = 'https://openalex.org/F4320335774'
last_dict['China Post-doctoral Science Foundation'] = 'https://openalex.org/F4320321543'
last_dict['Alzheimers Research UK'] = 'https://openalex.org/F4320321081'
last_dict['BLRD VA'] = 'https://openalex.org/F4320309842'
last_dict['Texas AM University'] = 'https://openalex.org/F4320310164'
last_dict['NSF CAREER award'] = 'https://openalex.org/F4320306076'
last_dict['DIVISION OF CANCER PREVENTION AND CONTROL'] = 'https://openalex.org/F4320309684'
last_dict['Fondo de Investigacion Sanitaria'] = 'https://openalex.org/F4320334923'
last_dict['MICINN (Spain)'] = 'https://openalex.org/F4320322930'
last_dict['Swiss NSF'] = 'https://openalex.org/F4320320924'
last_dict['Research Grant Council of Hong Kong'] = 'https://openalex.org/F4320321592'
last_dict['state budget of the Czech Republic'] = 'https://openalex.org/F4320321006'
last_dict['Solvay SA'] = 'https://openalex.org/F4320333783'

last_dict['PAPIIT'] = 'https://openalex.org/F4320335100'
last_dict['Beijing Municipal Science and Technology Project'] = 'https://openalex.org/F4320335843'
last_dict['Region Centre-Val de Loire'] = 'https://openalex.org/F4320325797'
last_dict['Programme of Introducing Talents of Discipline to Universities'] = 'https://openalex.org/F4320335785'
last_dict['National Science and Technology Support Program of China'] = 'https://openalex.org/F4320335961'
last_dict['Shenzhen Science and Technology Innovation Committee'] = 'https://openalex.org/F4320326705'
last_dict['ARC Future Fellowship'] = 'https://openalex.org/F4320334704'
last_dict['Guangdong Natural Science Funds for Distinguished Young Scholar'] = 'https://openalex.org/F4320327517'
last_dict['FWO-Vlaanderen'] = 'https://openalex.org/F4320321730'
last_dict['Hong Kong RGC'] = 'https://openalex.org/F4320321592'
last_dict['National Institute of Drug Abuse'] = 'https://openalex.org/F4320337347'
last_dict['Programa de Apoyo a Proyectos de Investigacion e Innovacion Tecnologica (PAPIIT)'] = 'https://openalex.org/F4320335100'


for k, v in last_dict.items():
    v = str(v)
    last_dict[k] = v.split('F')[-1]

last_dict = {k.strip():str(v).split('F')[-1] for k, v in last_dict.items()}

In [3]:
import json
with open('final_map.json', "w") as f:
    json.dump(last_dict, f)

In [ ]:
manual_annotations = HHS_map | p_1000 | new_p_1000 | last_dict 


In [1]:
import pandas as pd

test = pd.read_parquet('artifacts/oa_institutions.parquet')

In [7]:
test

,id,display_name,country_code,ids:ror,homepage_url,source,alternate_titles,acronyms
0,4320321001,National Natural Science Foundation of China,CN,https://ror.org/01h0zpd94,https://www.nsfc.gov.cn,funders,"[""Chinese National Science Foundation"", ""Gu\u0...",[NSFC]
1,4320332161,National Institutes of Health,US,https://ror.org/01cwqze88,http://www.nih.gov,funders,"[""Institutos Nacionales de la Salud"", ""NIH"", ""...",[NIH]
2,4320306076,National Science Foundation,US,https://ror.org/021nxhr62,https://www.nsf.gov,funders,"[""NSF"", ""National Science Foundation"", ""U.S. N...",[NSF]
3,4320334764,Japan Society for the Promotion of Science,JP,https://ror.org/00hhkn466,http://www.jsps.go.jp,funders,"[""Grants-in-Aid for Scientific Research"", ""Gra...",[JSPS]
4,4320320879,Deutsche Forschungsgemeinschaft,DE,https://ror.org/018mejw64,http://www.dfg.de/en,funders,"[""DFG"", ""Deutsche Forschungsgemeinschaft"", ""Ge...",[DFG]
...,...,...,...,...,...,...,...,...
117934,4405252777,Escuela Normal Superior del Estado,MX,https://ror.org/02kcab747,https://ense.mx,institutions,"[""ENSE"", ""Escuela Normal"", ""Escuela Normal Sup...",[ENSE]
117935,4405252781,Shijonawate Gakuen Junior College,JP,https://ror.org/036vefa56,https://jc.shijonawate-gakuen.ac.jp,institutions,"[""Shijonawate Gakuen Junior College"", ""Shij\u0...",[]
117936,4405252789,The Asian International School,VN,https://ror.org/0161zbh32,https://www.asianintlschool.edu.vn,institutions,"[""Asian International School"", ""The Asian Inte...",[]
117937,4405252801,Federation Regionale des Acteurs en Promotion ...,FR,https://ror.org/03cw6ep97,https://frapscentre.org,institutions,"[""FRAPS"", ""FRAPS CVL - F\u00e9d\u00e9ration R\...","[FRAPSCVL, FRAPS]"


In [28]:
u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(manual_annotations))

#u_agencies['id'] = u_agencies['id'].fillna(u_agencies['grant_agency'].map(min_100_map))


u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'manual'

print(u_agencies.shape)
print(u_agencies.id.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(7425486, 10)
0.9848124419061595


0.653524786617311

In [29]:
u_agencies['agency'] = u_agencies['display_name'].fillna(u_agencies['all_names_lower'])

### Prepare data for detecting substrings

In [30]:
found_agency = institutions.all_names.explode().dropna().unique()

found_agency = sorted(found_agency, key=len, reverse=True)

found_agency = [x for x in found_agency if  " " in x]    

found_agency_lower = [x.lower() for x in found_agency]

found_agency_lower_asc = sorted(found_agency_lower, key=len)

found_name = u_agencies['grant_agency']

found_name = sorted(found_name, key=len, reverse=True)

found_name = [x for x in found_name if  " " in x]

found_name_lower = [x.lower() for x in found_name]

found_name_lower_asc = sorted(found_name_lower, key=len)



In [31]:
def longest_string(s1, s2):
    if isinstance(s1, str) and isinstance(s2, str):
        return max(s1, s2, key=len)
    elif isinstance(s1, str):
        return s1
    elif isinstance(s2, str):
        return s2
    else:
        return np.nan

def build_automaton(words):
    A = ahocorasick.Automaton()
    for word in words:
        A.add_word(word, word)
    A.make_automaton()
    return A


def find_matches_end(text, automatons):
    matches = []
    for A in automatons:
        for end_index, word in A.iter(text):
            start_index = end_index - len(word) + 1
            # Check for word boundaries
            if end_index == len(text) - 1 and (start_index == 0 or not text[start_index - 1].isalnum()):
                matches.append(word)
        if matches:
            return max(matches, key=len, default=None)
    return None

def find_longest_match(text, automatons):
    matches = []
    for A in automatons:
        for end_index, word in A.iter(text):
            start_index = end_index - len(word) + 1
            # Check for word boundaries
            if (start_index == 0 or not text[start_index - 1].isalnum()) and \
            (end_index == len(text) - 1 or not text[end_index + 1].isalnum()):
                matches.append(word)
        if matches:
            return max(matches, key=len, default=None)
    return None


def find_matches_start(text, automatons):
    matches = []
    for A in automatons:
        for end_index, word in A.iter(text):
            if end_index - len(word) + 1 == 0 and (end_index + 1 == len(text) or not text[end_index + 1].isalnum()):
                matches.append(word)
        if matches:
            return max(matches, key=len, default=None)
    return None

def shortest_matching_substring(substrings, target_string):
    # Filter the substrings that are present in the target_string
    for s in substrings:
        if len(s) < len(target_string):
            continue
        if target_string in s:
            return s
    return None

## Detecting substrings

Here we detect the longest substrings in the start, end and anywhere in the names

In [ ]:
A = build_automaton(found_agency_lower)

u_agencies['agency_in_name_rev'] = u_agencies['grant_agency_lower'].progress_apply(lambda x: find_matches_end(x, [A]))
u_agencies['agency_in_name_start'] = u_agencies['grant_agency_lower'].progress_apply(lambda x: find_matches_start(x, [A]))
u_agencies['agency_in_name'] = u_agencies['grant_agency_lower'].progress_apply(lambda x: find_longest_match(x, [A]))

def split_sentences(sentences_list, num_splits=10):
    split_size = len(sentences_list) // num_splits
    return [sentences_list[i * split_size: (i + 1) * split_size] for i in range(num_splits)]



100%|██████████| 7425486/7425486 [00:24<00:00, 308220.32it/s]


In [ ]:
#rename agency in name
u_agencies.rename(columns={'agency_in_name': 'agency_in_name_search'}, inplace=True)


In [ ]:
u_agencies['agency_in_name'] = u_agencies[['agency_in_name_start', 'agency_in_name_rev']].apply(
    lambda x: longest_string(x['agency_in_name_start'], x['agency_in_name_rev']), axis=1)

### Substrings in the start or end of names

In [35]:
test = u_agencies[ u_agencies.id.isna() &
    ((u_agencies.agency_in_name.str.len() / u_agencies.grant_agency.str.len()) > 0.5)][['grant_agency', 'agency_in_name']]

In [36]:
u_agencies['agency_in_name'][((u_agencies.agency_in_name.str.len() / 
             u_agencies.grant_agency.str.len()) < 0.5)] = np.nan

/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_80004/1057552185.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  u_agencies['agency_in_name'][((u_agencies.agency_in_name.str.len() /


In [37]:
name_to_id = (u_agencies[['agency_in_name']].dropna()
              .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
                      left_on='agency_in_name', right_on='all_names_lower', how='inner')
                      .drop_duplicates()
)


u_agencies = u_agencies.merge(name_to_id[['agency_in_name', 'id']],
    left_on='agency_in_name', right_on='agency_in_name', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'start_or_end'

u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['agency_in_name'])


### Substrings anywhere in the names

In [39]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(8652268, 15)
0.7239342331975848 0.7240076243593009


0.7962735930093819

In [40]:
u_agencies['agency_in_name_search'][ u_agencies.id.isna() & ((u_agencies.agency_in_name_search.str.len()
                     / u_agencies.grant_agency.str.len()) < 0.5)] = np.nan

In [41]:
name_to_id = (u_agencies[['agency_in_name_search']].dropna()
              .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
                      left_on='agency_in_name_search', right_on='all_names_lower', how='inner')
                      .drop_duplicates()
)


u_agencies = u_agencies.merge(name_to_id[['agency_in_name_search', 'id']],
    left_on='agency_in_name_search', right_on='agency_in_name_search', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'substring'

u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['agency_in_name_search'])


In [43]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(8958031, 15)
0.673423434234599 0.6734838269704582


0.8085172948567778

### Matching acronyms

In [44]:
acronyms = (institutions.explode([ 'acronyms'])
                              [['acronyms', 'id']].dropna().drop_duplicates())

u_agencies = u_agencies.merge(acronyms,
    left_on='abbrev', right_on='acronyms', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'acronyms'

u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)

In [45]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(10994235, 16)
0.5011538319855815 0.7339567509699402


0.8366885867094713

### Document cluster ids
Based on the document clusters if we found and id we assign all documents in the cluster the same id

In [47]:
def infer_ids_by_component(u_agencies, component_col="component", id_col="id",
                           new_col="id_from_component", fill=False):
    """
    For rows with missing `id_col`, attach candidate IDs found on other rows
    sharing the same `component_col`. Duplicates are allowed (no dedup of result).
    
    Params
    ------
    u_agencies : pd.DataFrame
    component_col : str
        Column name grouping documents/components.
    id_col : str
        Column name for IDs to propagate.
    new_col : str
        Name of the attached candidate-ID column (was 'id_y' in your snippet).
    fill : bool
        If True, fill missing `id_col` with `new_col`.
    
    Returns
    -------
    pd.DataFrame
        Original frame plus `new_col`; optionally with `id_col` filled.
    """
    df = u_agencies.copy()
    df_reset = df.reset_index()  # keep original row index in a column named 'index'

    # Rows needing IDs
    missing = df_reset[df_reset[id_col].isna()][["index", component_col, id_col]]

    # Known IDs per component (allow multiple IDs per component)
    donors = (
        df[[component_col, id_col]]
        .dropna(subset=[id_col])
        .drop_duplicates([component_col, id_col])
    )

    # Attach candidate IDs to the missing rows via left-merge on component
    doc_clus = missing.merge(donors, on=component_col, how="left", suffixes=("_x", "_y"))

    # Join back to original df by the saved row index
    df = df.join(doc_clus[["index", f"{id_col}_y"]].set_index("index"), how="left")

    # Rename the attached column to something stable/user-friendly
    df.rename(columns={f"{id_col}_y": new_col}, inplace=True)

    # Optionally fill missing ids with the inferred ones
    if fill:
        df[id_col] = df[id_col].fillna(df[new_col])

    return df

u_agencies = infer_ids_by_component(u_agencies, fill=True)

u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'

u_agencies.drop(columns=['id_from_component'], inplace=True)

In [48]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(11138573, 16)
0.4875480009872001 0.7374042437931682


0.8495878362375567

### Extract funders with 1000 > papers associated

In [49]:
u_agencies_missing = u_agencies[u_agencies['id'].isna()]
u_agencies_missing.shape

(5430589, 16)

In [50]:
x = u_agencies_missing['counts'].cumsum()

test = u_agencies_missing[((u_agencies['counts'].sum()*1) - u_agencies[~u_agencies.id.isna()]['counts'].sum()) > x]
test = test[test.counts >= 1000]
test

,grant_agency,counts,grant_agency_lower,abbrev,component,display_name_lower,display_name,match_type,all_names_lower,agency,agency_in_name_rev,agency_in_name_start,agency_in_name_search,agency_in_name,acronyms,id
8471,"CereSpir, Inc.",1297,"cerespir, inc.",NaN,2253,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
8683,Cogstate,1227,cogstate,NaN,2363,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
10414,Municipality of Rotterdam,1007,municipality of rotterdam,NaN,2849,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN


#### Enhance missing funders with best guess from OpenAlex API

In [184]:
res_dict = {}
for abbrev in test.grant_agency.dropna().unique():
    res = query_openalex_funders(abbrev)[0]
    try:
        res_dict[abbrev] = res['id']
    except:
        pass

for k, v in res_dict.items():
    v = str(v)
    res_dict[k] = v.split('F')[-1]

In [ ]:
test['id'] = test['id'].fillna(test['grant_agency'].map(res_dict))


In [ ]:
# Write to file
#test[['grant_agency', 'id', 'display_name_y', 'all_names']].to_csv('new_plus_1000.csv', index=False)

### Extract funders with more than 100 papers using NER



In [54]:
x = u_agencies_missing['counts'].cumsum()

test = u_agencies_missing[((u_agencies['counts'].sum()*1) - u_agencies[~u_agencies.id.isna()]['counts'].sum()) > x]
test = test[(test.counts >= 100)]
test

,grant_agency,counts,grant_agency_lower,abbrev,component,display_name_lower,display_name,match_type,all_names_lower,agency,agency_in_name_rev,agency_in_name_start,agency_in_name_search,agency_in_name,acronyms,id
8471,"CereSpir, Inc.",1297,"cerespir, inc.",NaN,2253,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
8683,Cogstate,1227,cogstate,NaN,2363,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
10414,Municipality of Rotterdam,1007,municipality of rotterdam,NaN,2849,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
10635,Doctoral Program of Higher Education,998,doctoral program of higher education,NaN,2877,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
10636,Research Grants Council,997,research grants council,NaN,2878,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73285,Shenzhen Outstanding Talents Training Fund,100,shenzhen outstanding talents training fund,NaN,26601,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
73286,Beijing Top Discipline for Artificial Intellig...,100,beijing top discipline for artificial intellig...,NaN,26602,NaN,NaN,nan,NaN,NaN,university of science and technology beijing,None,NaN,NaN,NaN,NaN
73298,Strategic Cooperation Technology Project of CNPC,100,strategic cooperation technology project of cnpc,NaN,26604,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN
73299,Research Fund of Guangdong-Hong Kong-Macao Joi...,100,research fund of guangdong-hong kong-macao joi...,NaN,26605,NaN,NaN,nan,NaN,None,None,None,None,NaN,NaN,NaN


In [261]:
# Initialize GLiNER with the base model
model = GLiNER.from_pretrained("gliner_news_small", load_onnx_model=True, load_tokenizer=True, onnx_model_file="model_quantized.onnx")
model.to("mps")

def predict_entities(text, labels=['organisation', 'funder', 'funding agency', 'funding program']):
    entities = model.predict_entities(text, labels, threshold=0.3, flat_ner = False, multi_label=True)
    return [(entity["text"]) for entity in entities], [(entity["label"]) for entity in entities] 

test[['NER_agency', 'NER_label']] = test['grant_agency'].str.replace("-", " ").progress_apply(lambda x: pd.Series(predict_entities(x)))

config.json not found in /Users/jrk504/projects/openalex/gliner_news_small
100%|██████████| 8545/8545 [01:57<00:00, 72.56it/s]


In [262]:
test_ = test[['grant_agency', 'NER_agency', 'NER_label']].explode(['NER_agency', 'NER_label']).drop_duplicates(subset=['NER_agency', 'grant_agency'])
test_['NER_agency'] = test_['NER_agency'].fillna(test_['grant_agency'])

In [264]:
import math
import time
import typing as t
import threading
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ---------------------- Rate Limiter ---------------------- #

class TokenBucketLimiter:
    """Thread-safe token-bucket limiter (aggregate rate for all workers)."""
    def __init__(self, rps: float):
        self.capacity = max(float(rps), 0.1)
        self.tokens = self.capacity
        self.fill_rate = self.capacity
        self.last = time.monotonic()
        self.lock = threading.Lock()

    def acquire(self):
        """Block until 1 token is available."""
        while True:
            with self.lock:
                now = time.monotonic()
                elapsed = now - self.last
                self.last = now
                self.tokens = min(self.capacity, self.tokens + elapsed * self.fill_rate)
                if self.tokens >= 1.0:
                    self.tokens -= 1.0
                    return
                wait = (1.0 - self.tokens) / self.fill_rate
            time.sleep(wait)

# ---------------------- Helpers ---------------------- #

def _normalize(s: str) -> str:
    return "".join(ch for ch in (s or "").lower() if ch.isalnum() or ch.isspace()).strip()

def _get_with_limits(
    session: requests.Session,
    url: str,
    params: dict,
    limiter: TokenBucketLimiter,
    timeout: int = 30,
    max_retries: int = 6,
) -> dict:
    """GET with shared rate limit + exponential backoff and Retry-After support."""
    backoff = 1.0
    for attempt in range(max_retries):
        limiter.acquire()
        resp = session.get(url, params=params, timeout=timeout)
        if resp.status_code < 400:
            return resp.json()
        if resp.status_code in (429, 500, 502, 503, 504):
            ra = resp.headers.get("Retry-After")
            if ra:
                try:
                    delay = max(float(ra), backoff)
                except ValueError:
                    delay = backoff
            else:
                delay = backoff
            time.sleep(delay)
            backoff = min(backoff * 2, 16)
            continue
        resp.raise_for_status()
    limiter.acquire()
    resp = session.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def _score_funder(r: dict, *, norm_query: str, country_code: str | None) -> tuple:
    #names = [r.get("display_name", "")] + (r.get("alternate_titles") or [])
    exact =  0
    country_match = 1 if (country_code and (r.get("country_code") or "").upper() == country_code.upper()) else 0
    rel = float(r.get("relevance_score") or 0.0)
    wc = math.log1p(int(r.get("works_count") or 0))
    return (exact, country_match, rel, wc)

# ---------------------- Core Search ---------------------- #

def find_top_funder_fullsearch(
    query: str,
    *,
    email: str | None = None,
    country_code: str | None = None,
    session: requests.Session | None = None,
    limiter: TokenBucketLimiter | None = None,
    rps: float | None = None,
    per_page: int = 25,
    select_fields: t.Iterable[str] = (
        "id","display_name","alternate_titles","ids",
        "country_code","homepage_url","works_count",
        "cited_by_count","image_thumbnail_url","relevance_score"
    ),
    timeout: int = 30,
) -> dict | None:
    """Full OpenAlex /funders?search= query returning the top-ranked funder."""
    base = "https://api.openalex.org/funders"
    sess = session or requests.Session()
    if limiter is None:
        limiter = TokenBucketLimiter(rps if rps is not None else (10 if email else 1))

    params = {
        "search": query,
        "per-page": per_page,
        "select": ",".join(select_fields),
    }
    if email:
        params["mailto"] = email

    data = _get_with_limits(sess, base, params, limiter, timeout=timeout)
    results: list[dict] = data.get("results", [])
    if not results:
        return None

    norm_q = _normalize(query)
    return max(results, key=lambda r: _score_funder(r, norm_query=norm_q, country_code=country_code))

# ---------------------- Batched concurrent version ---------------------- #

def batch_find_top_funders_fullsearch(
    queries: list[str],
    *,
    email: str | None = None,
    country_code: str | None = None,
    max_workers: int = 8,
    rps: float | None = None,
    per_page: int = 25,
    timeout: int = 30,
    select_fields: t.Iterable[str] = (
        "id",
        "country_code","works_count",
        "relevance_score"
    ),
    show_progress: bool = True,
) -> list[dict | None]:
    """
    Concurrently resolve multiple funder names using /funders?search=.
    Displays tqdm progress bar if show_progress=True.
    """
    sess = requests.Session()
    effective_rps = rps if rps is not None else (10 if email else 1)
    limiter = TokenBucketLimiter(effective_rps)
    results: list[dict | None] = [None] * len(queries)

    def _task(i: int, q: str):
        try:
            results[i] = find_top_funder_fullsearch(
                q,
                email=email,
                country_code=country_code,
                session=sess,
                limiter=limiter,
                per_page=per_page,
                timeout=timeout,
                select_fields=select_fields,
            )
        except Exception:
            results[i] = None

    # tqdm progress bar
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_task, i, q) for i, q in enumerate(queries)]
        if show_progress:
            for _ in tqdm(as_completed(futures), total=len(futures), desc="Searching funders"):
                pass
        else:
            for _ in as_completed(futures):
                pass

    return results


In [265]:

results = batch_find_top_funders_fullsearch(
    test_['NER_agency'],
    email="jad@sodas.ku.dk",  # enter polite pool for 10 rps
    max_workers=3,
    show_progress=True
)

Searching funders: 100%|██████████| 14796/14796 [49:21<00:00,  5.00it/s] 


In [266]:
ids = [x.get('id') if x else None for x in results]
rels = [x.get('relevance_score') if x else None for x in results]
wc = [x.get('works_count') if x else None for x in results]

oa_results = pd.DataFrame({'id':ids, 'relevance_score':rels, 'works_count':wc}, index=test_.index)

In [267]:
df = (
    pd.concat([test_, oa_results], axis=1).reset_index()
 .pipe(lambda x: x.assign(id = x.id.str.extract("([0-9]+)").astype(float)))
 .merge(institutions[['id', 'display_name']], on='id', how='left')
 )[['index', 'NER_agency', 'grant_agency', 'display_name', 'id']].rename(columns={'id':'id_oa'})

df

,index,NER_agency,grant_agency,display_name,id_oa
0,8471,CereSpir,"CereSpir, Inc.",NaN,NaN
1,8471,"CereSpir, Inc.","CereSpir, Inc.",NaN,NaN
2,8683,Cogstate,Cogstate,NaN,NaN
3,10414,Municipality of Rotterdam,Municipality of Rotterdam,NaN,NaN
4,10635,Doctoral Program of Higher Education,Doctoral Program of Higher Education,Specialized Research Fund for the Doctoral Pro...,4.320336e+09
...,...,...,...,...,...
14791,73299,Research Fund,Research Fund of Guangdong-Hong Kong-Macao Joi...,Thailand Research Fund,4.320323e+09
14792,73299,Joint Laboratory for Intelligent Micro Nano Op...,Research Fund of Guangdong-Hong Kong-Macao Joi...,NaN,NaN
14793,73300,Chongqing Talents_ Exceptional Young Talents P...,Chongqing Talents_ Exceptional Young Talents P...,NaN,NaN
14794,73300,Chongqing Talents_,Chongqing Talents_ Exceptional Young Talents P...,NaN,NaN


In [51]:
#df.to_parquet('ner_agency_oa_v3.parquet')

df = pd.read_parquet('ner_agency_oa_v3.parquet')

In [52]:
# if id not na then "display_exact" use fillna approach
df = df.assign(NER_agency_lower = df.NER_agency.str.lower()).merge(institutions[['display_name_lower','display_name', 'id']],
    left_on='NER_agency_lower', right_on='display_name_lower', how='left')

df.loc[~df['id'].isna(), 'match_type'] = 'display_exact'

df = df.merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
    left_on='NER_agency_lower', right_on='all_names_lower', how='left')

df['id'] = df['id_x'].fillna(df['id_y'])
df.drop(columns=['id_x', 'id_y'], inplace=True)
df.loc[(~df['id'].isna()) & (df['match_type']=='nan'), 'match_type'] = 'all_exact'

In [53]:
df = df.drop_duplicates(subset=['NER_agency', 'id'])

In [54]:
df['agency_in_name'] = df['NER_agency'].str.lower().progress_apply(lambda x: find_longest_match(x, [A]))

df['agency_in_name'][ df.id.isna() & ((df.agency_in_name.str.len()
                     / df.NER_agency.str.len()) < 0.5)] = np.nan

name_to_id = (df[['agency_in_name']].dropna()
              .merge(institutions.explode(['all_names_lower'])[['all_names_lower', 'id']],
                      left_on='agency_in_name', right_on='all_names_lower', how='inner')
                      .drop_duplicates()
)


df = df.merge(name_to_id[['agency_in_name', 'id']],
    left_on='agency_in_name', right_on='agency_in_name', how='left')

df['id'] = df['id_x'].fillna(df['id_y'])
df.drop(columns=['id_x', 'id_y'], inplace=True)
df.loc[(~df['id'].isna()) & (df['match_type']=='nan'), 'match_type'] = 'substring'


100%|██████████| 11619/11619 [00:00<00:00, 132259.39it/s]
/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_80004/994215213.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['agency_in_name'][ df.id.isna() & ((df.agency_in_name.str.len()


In [55]:
df = df.drop_duplicates(subset=['NER_agency', 'id'])

In [56]:
df['id'] = df['id'].fillna(df['id_oa'])

df.loc[(~df['display_name_x'].isna()) & (df['match_type']=='nan'), 'match_type'] = 'oa'

In [57]:
#df[['index', 'NER_agency', 'grant_agency','match_type', 'id']].merge(institutions[['display_name', 'id', 'country_code']], on='id', how='left')

In [58]:
u_agencies = (u_agencies.join(df[['index', 'id', 'NER_agency', 'match_type']].dropna(subset='id').set_index('index'),
                               how='left', rsuffix="_100")
 .pipe(lambda x: x.assign(id = x.id.fillna(x.id_100),
                          match_type = x.match_type.fillna(x.match_type_100)))
 .drop(columns=['id_100', 'NER_agency', 'match_type_100'], inplace=False)
                          )


In [59]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()


(11142809, 16)
0.48705986075862917 0.7375040710111787


0.8620176656668821

In [39]:
def summarize_u_agencies(u_agencies):
    """
    Summarize ID coverage in u_agencies by count ranges.

    Prints:
      - overall share of missing ids
      - overall share of missing agency names
      - share of counts with matched ids
      - missing id rate by count ranges
    """
    t = u_agencies.drop_duplicates(subset=['grant_agency_lower'])
    
    print("Overall missing id rate:", t.id.isna().mean())
    #print("Overall missing agency rate:", t.agency.isna().mean())

    coverage = t.loc[t.id.notna(), 'counts'].sum() / t['counts'].sum()
    print("Share of total counts covered by matched ids:", coverage)

    bins = [
        (1000, float('inf'), "≥1000"),
        (100, 1000, "100–1000"),
        (10, 100, "10–100"),
        (1, 10, "1–10")
    ]
    for low, high, label in bins:
        subset = t[(t['counts'] >= low) & (t['counts'] < high)]
        if not subset.empty:
            print(f"Missing id rate ({label}):", subset.id.isna().mean())
            print(f"Matched strings ({label}):", (~subset.id.isna()).sum())
            print(f"Total strings ({label}):", len(subset))


In [ ]:
summarize_u_agencies(u_agencies)


In [61]:
u_agencies = infer_ids_by_component(u_agencies, fill=True)

u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'

u_agencies.drop(columns=['id_from_component'], inplace=True)

In [62]:
summarize_u_agencies(u_agencies)


Overall missing id rate: 0.7325685902285769
Overall missing agency rate: 0.8153830081174891
Share of total counts covered by matched ids: 0.7338796285100674
Missing id rate (≥1000): 0.001034126163391934
Missing id rate (100–1000): 0.21265780592475314
Missing id rate (10–100): 0.5910365953182589
Missing id rate (1–10): 0.7398676417112187


In [2]:
mm = pd.read_parquet('mid_missing.parquet')

In [4]:
inst_norm = pd.read_parquet('artifacts/institutions_normalized.parquet')

In [11]:
inst_norm[inst_norm.all_names_lower.apply(lambda x: any([y=='nsf' for y in x]))]

,id,display_name,country_code,ids:wikidata,ids:ror,ids:crossref,ids:doi,description,homepage_url,alternate_titles,...,grants_count,works_count,cited_by_count,other_ids:ror,other_ids:wikidata,homepage_domain,all_names,all_names_lower,display_name_lower,source
2,4320306076,National Science Foundation,US,https://www.wikidata.org/entity/Q304878,https://ror.org/021nxhr62,100000001,https://doi.org/10.13039/100000001,United States government agency,https://www.nsf.gov,"[""NSF"", ""National Science Foundation"", ""U.S. N...",...,207831,391743,13017807,,,nsf.gov,"[USNSF, USA NSF, National Science Foundation, ...","[usnsf, national science foundation, nsf, us n...",national science foundation,funders
51,4320335353,National Science Foundation of Sri Lanka,LK,https://www.wikidata.org/entity/Q6978282,https://ror.org/010xaa060,501100008982,https://doi.org/10.13039/501100008982,successor to the Natural Resources Energy & Sc...,https://www.nsf.gov.lk,"[""NSF"", ""National Science Foundation"", ""Nation...",...,16632,15879,427921,,,nsf.gov.lk,[Natural Resources Energy and Science Authorit...,"[national science foundation of sri lanka, ජාත...",national science foundation of sri lanka,funders
308,4320307381,National Sleep Foundation,US,https://www.wikidata.org/entity/Q12238011,https://ror.org/00zc1hf95,100003187,https://doi.org/10.13039/100003187,organization,http://sleepfoundation.org,"[""NSF"", ""National Sleep Foundation"", ""The Nati...",...,1706,3395,90398,,,sleepfoundation.org,"[The National Sleep Foundation, NSF, National ...","[the national sleep foundation, nsf, national ...",national sleep foundation,funders
440,4320320373,National Stroke Foundation,AU,https://www.wikidata.org/entity/Q30290741,https://ror.org/004ckc033,501100000930,https://doi.org/10.13039/501100000930,"nonprofit organization in Melbourne, Australia",https://strokefoundation.com.au,"[""NSF"", ""National Stroke Foundation""]",...,1052,910,30572,,,strokefoundation.com.au,"[National Stroke Foundation, NSF]","[national stroke foundation, nsf]",national stroke foundation,funders
701,4320322435,Norsk Sykepleierforbund,NO,https://www.wikidata.org/entity/Q11992400,https://ror.org/01822d048,501100004190,https://doi.org/10.13039/501100004190,"organization in Oslo, Norway",https://www.nsf.no/english,"[""NSF"", ""Norsk Sykepleierforbund"", ""Norwegian ...",...,590,499,18266,,,nsf.no,"[Norwegian Nurses Organisation, NSF, Norsk Syk...","[norsk sykepleierforbund, norwegian nurses org...",norsk sykepleierforbund,funders
15493,4320321843,Bulgarian National Science Fund,BG,,https://ror.org/05eg49r29,501100003336,https://doi.org/10.13039/501100003336,,http://www.fni.bg,"[""BNSF"", ""BSF"", ""Bulgarian National Science Fu...",...,2006,3556,48245,,,fni.bg,"[National Science Fund, BSF, Bulgarian Scienti...","[bulgarian national science fund, bulgarian sc...",bulgarian national science fund,funders
15881,4320330889,Neurosciences Foundation,GB,,https://ror.org/01fy9e204,501100020414,https://doi.org/10.13039/501100020414,,https://neurosciencesfoundation.org.uk,"[""NSF"", ""Neurosciences Foundation"", ""The Neuro...",...,63,102,1389,,,neurosciencesfoundation.org.uk,"[Neurosciences Foundation, The Neurosciences F...","[the neurosciences foundation, neurosciences f...",neurosciences foundation,funders
25053,4320319363,National Scleroderma Foundation,US,,,100019921,https://doi.org/10.13039/100019921,,,"[""NATIONAL SCLERODERMA FOUNDATION, INC."", ""NSF...",...,18,49,250,,,,"[National Scleroderma Fndn., NATIONAL SCLERODE...","[national scleroderma foundation, sf, national...",national scleroderma foundation,funders
25087,4320316730,Nick Simons Foundation,US,,,100016620,https://doi.org/10.13039/100016620,,,"[""NSF"", ""Nick Simons Foundation"", ""The Nick Si...",...,18,25,225,,,,"[Nick Simons Foundation, The Nick Simons Found...","[the nick simons foundation, nsf, nick simons ...",nick simons foundation,funders
106526,4210136569,Norsk Sveiseteknisk Forbund,NO,,https://ror.org/03yr3hm52,,,,http://www.

In [3]:
mm

,index,grant_agency,counts,grant_agency_lower,abbrev,component,match_type,display_name_lower,display_exact_name,id,agency_in_name,agency_in_name_search,acronyms,key_lower,NER_agency,display_name_lower_disp,all_names_lower_alt,match_substring
0,6327,Doctoral Program of Higher Education,998,doctoral program of higher education,None,2940,None,None,None,NaN,None,None,None,doctoral program of higher education,Doctoral Program of Higher Education,None,None,None
1,6328,Research Grants Council,997,research grants council,None,2941,None,None,None,NaN,None,None,None,research grants council,Research Grants Council,None,None,None
3,6339,Program for Innovative Research Team in Univer...,994,program for innovative research team in univer...,None,2950,None,None,None,NaN,None,None,None,program,Program,None,None,None
4,6339,Program for Innovative Research Team in Univer...,994,program for innovative research team in univer...,None,2950,None,None,None,NaN,None,None,None,innovative research team,Innovative Research Team,None,None,None
7,6339,Program for Innovative Research Team in Univer...,994,program for innovative research team in univer...,None,2950,None,None,None,NaN,None,None,None,university of ministry of education of china,University of Ministry of Education of China,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34902,50536,Strategic Cooperation Technology Project of CNPC,100,strategic cooperation technology project of cnpc,None,27421,None,None,None,NaN,None,None,None,strategic cooperation technology project,Strategic Cooperation Technology Project,None,None,None
34907,50537,Research Fund of Guangdong-Hong Kong-Macao Joi...,100,research fund of guangdong-hong kong-macao joi...,None,27422,None,None,None,NaN,None,None,None,joint laboratory for intelligent micro nano op...,Joint Laboratory for Intelligent Micro Nano Op...,None,None,None
34908,50538,Chongqing Talents_ Exceptional Young Talents P...,100,chongqing talents_ exceptional young talents p...,None,27423,None,None,None,NaN,None,None,None,chongqing talents_ exceptional young talents p...,Chongqing Talents_ Exceptional Young Talents P...,None,None,None
34909,50538,Chongqing Talents_ Exceptional Young Talents P...,100,chongqing talents_ exceptional young talents p...,None,27423,None,None,None,NaN,None,None,None,chongqing talents_,Chongqing Talents_,None,None,None


# Deduplication using LSH

In [63]:
u_agencies_missing = u_agencies[u_agencies['id'].isna()]
u_agencies_missing.shape

(5420338, 16)

In [64]:
u_agencies_exact = u_agencies.dropna(subset=['id','agency'])
print(u_agencies_exact.shape)
u_agencies_exact = u_agencies_exact.drop_duplicates(subset=['id', 'agency'])
print(u_agencies_exact.shape)

(2924942, 16)
(163767, 16)


## this block takes about about 1-2 hours to run

In [76]:
minhash_file = 'institutions_oa_minhashes.pkl'

cands = institutions.all_names_lower.explode()

# Serialize MinHashes and store them to a file
with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(cands.index, cands), total=len(cands), desc="Adding MinHashes to LSH"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)

Adding MinHashes to LSH: 100%|██████████| 326219/326219 [03:33<00:00, 1528.79it/s]


In [284]:

minhash_file = 'institutions_cand_minhashes.pkl'

queries = u_agencies_exact.agency.str.lower()
cands = u_agencies_missing.grant_agency.str.lower()

# Serialize MinHashes and store them to a file
with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(cands.index, cands), total=len(cands), desc="Adding MinHashes to LSH"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)


minhash_file = 'institutions_query_minhashes.pkl'

with open(minhash_file, 'wb') as f_out:
    for idx, text in tqdm(zip(queries.index, queries), total=len(queries), desc="Querying Candidate Strings"):
        lean_minhash = create_minhash(text)
        # Write a tuple of (index, MinHash) to the file
        pickle.dump((idx, lean_minhash), f_out)



Querying Candidate Strings: 100%|██████████| 163767/163767 [01:44<00:00, 1574.14it/s]


In [65]:

# Create the LSH object with threshold for near-duplicates
lsh = MinHashLSH(threshold=0.9, num_perm=128)
minhash_file = 'institutions_cand_minhashes.pkl'

# Load MinHashes from file and insert them into the LSH
ref_minhashes = {}
i = 0
with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            lsh.insert(f"ref_{idx}", lean_minhash)
            i += 1
    except EOFError:
        pass 

minhash_file = 'institutions_query_minhashes.pkl'

candidate_results = []
with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            result = lsh.query(lean_minhash)
            candidate_results.append({'cand_id': idx, 'duplicates': result})
    except EOFError:
        pass

In [ ]:
# with open("lsh.pkl", "wb") as f:
#     pickle.dump(lsh, f)

Writing:   2%|▏         | 91.2M/5.97G [00:00<00:33, 175MB/s]

Writing: 5.97GB [00:51, 116MB/s]                             


In [65]:
with open("lsh.pkl", "rb") as f:
    lsh = pickle.load(f)

In [77]:
minhash_file = 'institutions_oa_minhashes.pkl'

candidate_results_ = []
with open(minhash_file, 'rb') as f_in:
    try:
        while True:
            idx, lean_minhash = pickle.load(f_in)
            result = lsh.query(lean_minhash)
            candidate_results_.append({'cand_id': idx, 'duplicates': result})
    except EOFError:
        pass

In [ ]:
def create_set(text):
    if not isinstance(text, str):
        return set()
    s = set()
    for word in get_shingles(text, 3, set_range=False, wildcard=False):
        s.update(word.encode('utf-16', 'surrogatepass').decode('utf-16').encode('utf-8'))
    return s

def jaccard(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0


results_df = pd.DataFrame(candidate_results_)
results_df['duplicates'] = results_df['duplicates'].apply(lambda x: [int(y.split('_')[1]) for y in x])


In [81]:
results_df = pd.concat([results_df, institutions.explode("all_names_lower").reset_index()['all_names_lower']], axis=1)
results_df['duplicates_text'] = results_df['duplicates'].apply(lambda x: [u_agencies_missing['grant_agency'][int(i)] for i in x])
results_df = results_df.explode(['duplicates_text', 'duplicates']).dropna(subset=['duplicates_text'])


In [82]:
results_df = results_df.drop_duplicates(subset=['cand_id', 'duplicates'])
results_df = results_df.merge(institutions[['id', 'display_name']], left_on='cand_id', right_index=True, how='left')

In [95]:

results_df['jaccard'] = results_df.progress_apply(lambda row: jaccard(create_set(row['all_names_lower']), create_set(row['duplicates_text'].lower())), axis=1)

100%|██████████| 50306/50306 [00:03<00:00, 16193.17it/s]


In [100]:
results_df = results_df.query('jaccard>=0.9')

In [102]:
u_agencies = u_agencies.merge(results_df[['duplicates_text','id']],
    left_on='grant_agency', right_on='duplicates_text', how='left')

u_agencies['id'] = u_agencies['id_x'].fillna(u_agencies['id_y'])
#u_agencies['agency'] = u_agencies['agency_x'].fillna(u_agencies['agency_y'])

u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'deduplication'

u_agencies.drop(columns=['id_x', 'id_y'], inplace=True)
#u_agencies.drop(columns=['agency_x', 'agency_y'], inplace=True)

In [103]:
u_agencies = u_agencies.drop_duplicates(subset=['grant_agency_lower', 'id'])
print(u_agencies.shape)
print(u_agencies.id.isna().mean(), u_agencies.agency.isna().mean())
u_agencies[~u_agencies.id.isna()]['counts'].sum() / u_agencies['counts'].sum()

(11170061, 17)
0.48227408964015506 0.7381444917803045


0.8647583457624884

In [104]:
u_agencies = infer_ids_by_component(u_agencies, fill=True)

u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'

u_agencies.drop(columns=['id_from_component'], inplace=True)

summarize_u_agencies(u_agencies)


Overall missing id rate: 0.7277250190091047
Overall missing agency rate: 0.8153830081174891
Share of total counts covered by matched ids: 0.7377667787333896
Missing id rate (≥1000): 0.001034126163391934
Missing id rate (100–1000): 0.20549143785675597
Missing id rate (10–100): 0.5792885909606793
Missing id rate (1–10): 0.7352858138613972


In [ ]:
# component_id_map = u_agencies.groupby('component')['id'].first()
# u_agencies['id'] = u_agencies['id'].fillna(u_agencies['component'].map(component_id_map))
# u_agencies.loc[(~u_agencies['id'].isna()) & (u_agencies['match_type']=='nan'), 'match_type'] = 'doc_clustering'
# component_name_map = u_agencies.groupby('component')['agency'].first()
# u_agencies['agency'] = u_agencies['agency'].fillna(u_agencies['component'].map(component_name_map))

In [105]:
print(u_agencies.drop_duplicates('grant_agency').match_type.value_counts())

match_type
nan               5384500
start_or_end      1049038
acronyms           522745
substring          231125
doc_clustering      91996
all_exact           43021
display_exact       42707
deduplication       33307
manual                647
Name: count, dtype: int64


In [106]:
# id column to string to int
u_agencies['id'] = u_agencies['id'].astype('float64').astype('Int64')

In [111]:
institutions['source'] = institutions.sources_merged.apply(lambda x: ast.literal_eval(x)[0])

In [112]:
final_map = (u_agencies.merge(
        institutions[['id', 'display_name', 'ids:ror', 'source']], left_on='id', right_on='id', how='left')
[['grant_agency', 'id', 'counts', 'ids:ror', 'source', 'match_type', 'display_name_y']]).drop_duplicates()

In [113]:
final_map = final_map.assign(match_type = final_map.match_type.replace('nan', None))

In [116]:
final_map.drop_duplicates('grant_agency').query("(~match_type.isna()) & display_name_y.isna()")

,grant_agency,id,counts,ids:ror,source,match_type,display_name_y
1074,Canadian Foundation for Innovation,4320320995,6761,NaN,NaN,manual,NaN
2043,European Union's Horizon 2020 research and inn...,4321443650,3616,NaN,NaN,manual,NaN
8814,NSF-DMR,4320337367,758,NaN,NaN,doc_clustering,NaN
9222,Marie Curie Actions,4321443650,724,NaN,NaN,manual,NaN
14047,European Union's Horizon research and innovati...,4321443650,461,NaN,NaN,doc_clustering,NaN
...,...,...,...,...,...,...,...
11039395,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN
11068774,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN
11085166,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN
11093336,European Union's Horizon 2020 research and inn...,4321443650,1,NaN,NaN,doc_clustering,NaN


In [117]:
print((~final_map.drop_duplicates('grant_agency').isna()).sum())

grant_agency      7399086
id                2014586
counts            7399086
ids:ror           2014293
source            2014293
match_type        2014586
display_name_y    2014293
dtype: int64


In [118]:

# Load the data file (replace 'filename.csv' with the path to your file)
filename = 'v1.72-2025-10-06-ror-data_schema_v2.csv'

# Read the CSV file into a pandas DataFrame
ror = pd.read_csv(
    filename,
    dtype=str,  # Treat all columns as strings to handle mixed data types
    parse_dates=[
        'admin.created.date', 
        'admin.last_modified.date'
    ],  # Automatically parse date columns
    keep_default_na=False  # Prevent empty strings from being converted to NaN
)


In [119]:
in_dimensions = pl.read_parquet('funders_dimensions.parquet').with_columns(pl.lit('True').alias('in_dimensions')).to_pandas()
ror = ror.merge(in_dimensions, how='left', left_on="external_ids.type.grid.preferred", right_on='funder')

final_map = final_map.merge(ror[['id', 'types', 'relationships','funder', 'in_dimensions']], left_on="ids:ror", right_on='id', how='left', suffixes=("","_ror"))

In [120]:
final_map[final_map.id.isna()].to_feather('missing_agencies.feather')

In [121]:
final_map[~(final_map.id.isna())].to_feather('institutions_processed_duplicates.feather')

In [12]:
import pandas as pd

In [2]:
final_map = pd.read_feather('artifacts/institutions_processed_duplicates.feather')

final_map_missing = pd.read_feather('artifacts/missing_agencies.feather')

In [21]:
final_map = pd.read_parquet('artifacts/final_map.parquet')

In [1]:
import pandas as pd
final_map = pd.read_parquet('wos_funders.parquet')

In [6]:
import json

schema = final_map.dtypes.apply(lambda x: x.name).to_dict()
print(json.dumps(schema, indent=2))

{
  "grant_agency": "object",
  "id": "Int64",
  "counts": "uint32",
  "ids:ror": "object",
  "source": "object",
  "match_type": "object",
  "display_name": "object"
}


In [9]:
final_map[['grant_agency', 'id', 'counts', 'ids:ror', 'source', 'match_type', 'display_name']].to_parquet('wos_funders.parquet')

In [15]:
final_map = final_map.groupby('grant_agency').agg({'id' : list,
                                       'counts' : min,
                                       'ids:ror' : list,
                                       'source': list,
                                       'match_type':list,
                                       'display_name':list})

/var/folders/y6/k76sh60x15b90z0gvgbfk3xc0000gn/T/ipykernel_50304/3958644203.py:1: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  final_map = final_map.groupby('grant_agency').agg({'id' : list,


In [19]:
final_map.sort_values('counts', ascending=False).to_parquet('wos_funders_list.parquet')

In [22]:
(final_map[['grant_agency', 'id', 'counts', 'ids:ror', 'source', 'match_type', 'display_name']]
 .drop_duplicates('grant_agency')
 .notna().sum())

grant_agency    7399086
id              1952118
counts          7399086
ids:ror         1952011
source          1952011
match_type      1952118
display_name    1952011
dtype: int64

In [60]:
print(final_map.drop_duplicates('grant_agency').match_type.value_counts())

match_type
substring         1049258
acronyms           522745
substring_any      231125
all_exact           43653
display_exact       42999
doc_clustering      42055
deduplication       19582
manual                701
Name: count, dtype: int64


In [46]:
final_map.drop_duplicates('grant_agency').shape[0] + final_map_missing.shape[0]

7399086

In [61]:
summarize_u_agencies(pd.concat([final_map,
                                 final_map_missing]).pipe(
                                     lambda x: x.assign(grant_agency_lower = x.grant_agency.str.lower())
                                 ))


Overall missing id rate: 0.7361676834138703
Share of total counts covered by matched ids: 0.7213108823016756
Missing id rate (≥1000): 0.001034126163391934
Matched strings (≥1000): 2898
Total strings (≥1000): 2901
Missing id rate (100–1000): 0.29602933211116206
Matched strings (100–1000): 16896
Total strings (100–1000): 24001
Missing id rate (10–100): 0.5983852662185091
Matched strings (10–100): 105805
Total strings (10–100): 263449
Missing id rate (1–10): 0.7430599115032421
Matched strings (1–10): 1826519
Total strings (1–10): 7108735
